# 🚀 دفتر `main` — تجميع المشروع (البيانات ← النموذج ← التدريب ← التقييم)

هذا الدفتر هو نقطة الدخول الوحيدة لتشغيل المشروع كاملاً. لا يُعرِّف منطقاً
جديداً بنفسه — **يستورد** الدفاتر الأربعة الأخرى (`crypto_data_pipeline_v6`،
`model_v2 (1)`، `trainer_framework_v2`، `chicks_v4_5_input_output_patterns`)
عبر `%run`، ويربط مخرَج كل واحد بمدخل التالي.

## لماذا احتاج الأمر طبقة "ربط" صريحة (لا استيراد مباشر فحسب)

تحليل الدفاتر الأربعة، ثم تشغيلها فعلياً على بيانات حقيقية من Drive، كشفا
نقاط عدم تطابق حقيقية في العقد بينها — ليست أخطاء برمجية، بل فروقاً في
التصميم نشأت لأن كل دفتر طُوِّر بمعزل عن الآخر:

1. **أُطر زمنية**: خط الأنابيب يدعم عدة أطر (`tf_order`)، لكن النموذج
   (`build_nig_timenet_v2`) يقبل مدخلاً واحداً فقط `(seq_len, n_features)`
   لفريم واحد. الحل هنا: القسم ٣ يشتقّ فريماً واحداً فعلياً من البيانات
   المحمَّلة (`dataset['timeframes']`)، لا افتراضاً مُثبَّتاً بالكود.

2. **تسمية الأهداف**: بيانات خط الأنابيب تحمل مفاتيح `y` بصيغة
   `y_{هدف}_{class|reg}` (مثلاً `y_close_reg`)، بينما مخرجات النموذج
   بصيغة `y_{هدف}` (بلا `_reg`، والاسم نفسه هو متوسط NIG لا خام مُقيَّس).
   `trainer_framework` يحسم هذا فعلاً بتصميمه: `true_key` (مفتاح البيانات)
   و`output_keys` (مفاتيح مخرجات النموذج) مساحتا تسمية منفصلتان تماماً —
   القسم ٥ هنا يربطهما صراحةً بدل افتراض تطابقهما.

3. **مقياس الهدف**: `reg_target_mode` الافتراضي في خط الأنابيب هو
   `'return'` — أي أن `y_close_reg` عائد نسبي مباشر (`(مستقبلي/آخر_سعر) - 1`)
   لا قيمة مُقيَّسة بـ`base_params` (تلك دلالة `'window_scale'` القديمة
   فقط). لكن دالة فكّ التشفير في `chicks` (`decode_predictions_v4`) مبنية
   على افتراض `pred_real = raw*iqr + median` — صيغة `base_params` القديمة.
   القسم ٦ هنا يمرّر `base_params = [آخر_سعر, آخر_سعر]` بدل
   `dataset['base_params']` الخام، فتُصبح نفس الصيغة تكافئ رياضياً عكس
   العائد المباشر (`آخر_سعر × (١ + عائد)`) — مطابقة تماماً لما تفعله
   `invert_reg_predictions` في خط الأنابيب لنفس الوضع.
   **بلا هذا التصحيح، كل تقرير من `chicks` كان سيُفسِّر عائداً صغيراً
   (~0.01) على أنه سعر مطلق — خطأ صامت لا يظهر كاستثناء.**

4. **ترميز تصنيف الاتجاه**: خط الأنابيب يُشفِّر رأس `_class` بقيم
   **`+1.0`/`-1.0`** (`prepare_single_asset`، بروح رؤوس `tanh` القديمة في
   `model_v2` الأصلي — انظر `own_future[t] > own_last[t]`)، بينما
   `binary_classification` الجديد في `model_v2` يُخرج احتمالاً بعد
   `sigmoid` (نطاق `[0,1]`)، و`classification_task_loss` في
   `trainer_framework` يستدعي `binary_crossentropy`/يقارن بعتبة `0.5` —
   كلاهما يفترض عمداً تسمية `{0,1}` القياسية. **اكتُشف هذا فقط عند التدريب
   الفعلي على بيانات حقيقية**: الخسارة تُحسَب بصيغة رياضية خاطئة لقيم
   `-1`، والدقّة المُبلَّغة أثناء التدريب صفر دائماً تقريباً (`pred∈{0,1}`
   لا يُطابق أبداً `true=-1`). القسم ٥ هنا يحوِّل `(y+1)/2` قبل التغذية —
   نقطة التحويل الوحيدة، فأي جهة أخرى (النموذج، `trainer_framework`، خط
   الأنابيب) تبقى بلا تعديل.

كل هذه القرارات مُختبَرة فعلياً — نظرياً ببيانات تركيبية في القسم الأخير من
هذا الدفتر، وعملياً على بيانات حقيقية من Drive (خط أنابيب كامل + تدريب +
`chicks` + تقرير تصنيف، على 5 عملات حقيقية) قبل دمج هذه النسخة.


## ١) تحميل وتشغيل خط الأنابيب

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/drive/MyDrive/crypto"  # ⚠️ غيّره إن كانت دفاترك في مسار آخر على Drive

%cd {REPO_DIR}

In [ ]:
# يُعرِّف: CONFIG، build_dataset*، split_data، rolling_splits، mount_drive،
# save_data_to_drive/load_data_from_drive، invert_reg_predictions، ...
# ويُشغِّل تلقائياً run_pipeline_selftests() (بيانات تركيبية، بلا شبكة/Drive
# — آمن دائماً، ثوانٍ معدودة).
%run "crypto_data_pipeline_v6.ipynb"

## ٢) إعدادات هذا المشروع

عدّل هنا فوق افتراضيات خط الأنابيب — لا تُعدِّل `DEFAULT_CONFIG` مباشرة.
تحديد `tf_order`/`model_tf` مؤجَّل عمداً للقسم التالي (بعد تحميل البيانات
الفعلية)، لأن الفريم الصحيح يعتمد على ما بُني فعلاً في `dataset`، لا على
تخمين هنا.

⚠️ **ملاحظة**: تعديل `enabled_heads` هنا **لن** يُغيِّر بيانات مُحمَّلة
مسبقاً من Drive — ذلك الإعداد يؤثّر فقط على `build_dataset` وقت بنائه
الفعلي (في دفتر خط الأنابيب نفسه، قبل الحفظ). القسم ٥ هنا يفترض أن
البيانات المحمَّلة تحمل مفاتيح `_class` و`_reg` معاً لكل هدف (الافتراضي
في خط الأنابيب) — إن كانت بياناتك بُنيت بـ`enabled_heads` مختلف، عدّل
`build_target_configs` هناك ليطابقها.


In [ ]:
update_config({
    "project_name": "crypto_model",
    # أضف أي إعداد آخر يخصّ مشروعك هنا (مثلاً excluded_coins، eval_batch_size).
})
print("project_name:", CONFIG["project_name"])

## ٣) تحميل البيانات الجاهزة (من Drive) وتحديد الفريم الزمني الفعلي

In [ ]:
# اسم ملف البيانات: "preprocessing_output" (اليومي) أو HOURLY_DATA_NAME (فريم الساعة —
# القسم 20-ب في crypto_data_pipeline_v6: نافذة 168، stride=32، أفق 4).
DATA_FILENAME_BASE = "preprocessing_output"
dataset = load_data_from_drive(filename_base=DATA_FILENAME_BASE)  # يقرأ {DATA_FILENAME_BASE}_latest.pkl.gz

# النموذج الحالي (دفتر model_v2) يقبل فريماً واحداً فقط. نختار الفريم
# فعلياً من dataset['timeframes'] بدل تثبيته هنا: model_tf إن وُجد فعلاً
# في البيانات المحمَّلة، وإلا base_tf المُستخدَم فعلاً وقت البناء.
MODEL_TF = CONFIG.get("model_tf") if CONFIG.get("model_tf") in dataset["timeframes"] else dataset["base_tf"]
update_config({"tf_order": [MODEL_TF], "base_tf": MODEL_TF,
               # من البيانات نفسها لا من افتراضات CONFIG (اليومي: 32/1/1، الساعة: 168/32/4)
               "window_sizes": dict(dataset["window_sizes"]),
               "forecast_horizon": dataset.get("forecast_horizon", CONFIG["forecast_horizon"]),
               "stride": dataset.get("stride", CONFIG["stride"])})

print(f"الأطر المتوفّرة في البيانات: {dataset['timeframes']} — الفريم المُستخدَم فعلياً: {MODEL_TF}")
print(f"عدد الميزات: {len(dataset['feature_order'])} — طول النافذة: {dataset['window_sizes'][MODEL_TF]}"
      f" — الأفق: {CONFIG['forecast_horizon']} — stride: {CONFIG['stride']}")

In [ ]:
train, val, test = split_data(dataset, config=CONFIG)
print("train:", train[f"X_{MODEL_TF}"].shape, "| val:", val[f"X_{MODEL_TF}"].shape,
      "| test assets:", list(test.keys()) if isinstance(test, dict) and all(isinstance(v, dict) and "y" in v for v in test.values()) else test[f"X_{MODEL_TF}"].shape)

## ٤) النموذج — `build_nig_timenet_v2` من دفتر `model_v2`

In [ ]:
# يُعرِّف: HEAD_REGISTRY، MODEL_CONFIG، build_model_fn، build_nig_timenet_v2، ...
# ويُشغِّل تلقائياً run_model_selftests() (بلا بيانات حقيقية — آمن دائماً).
%run "model_v2 (1).ipynb"

In [ ]:
SEQ_LEN = dataset["window_sizes"][MODEL_TF]
N_FEATURES = len(dataset["feature_order"])
PRICE_TARGETS = list(CONFIG["targets"])  # افتراضياً ['high', 'low', 'close']


def model_builder():
    """صفر-وسيط، كما يتطلّب build_training_system — نفس المعمارية كل مرّة
    (ضروري لصحّة استئناف الحالة المحفوظة)."""
    return build_model_fn(SEQ_LEN, N_FEATURES)


model = model_builder()
model.summary()

## ٥) التدريب — ربط مخرجات النموذج بمفاتيح بيانات خط الأنابيب

`true_key`: مفتاح الهدف في `train['y']` (من خط الأنابيب، بصيغة
`y_{هدف}_reg`/`y_{هدف}_class`). `output_keys`: أسماء مخرجات النموذج
الفعلية (من دفتر `model_v2`، بصيغة `y_{هدف}`/`_nu`/`_alpha`/`_beta`/
`_confidence` للانحدار، و`y_{هدف}_class_logits` للتصنيف). مساحتا تسمية
منفصلتان تماماً في تصميم `trainer_framework` — هذه الدالة هي مكان الربط
الوحيد، فتعديل تسمية أي طرف مستقبلاً يحتاج تعديلاً هنا فقط.

رؤوس التصنيف الثنائي مُفعَّلة الآن افتراضياً في `model_v2` (`MODEL_CONFIG['head_types']`)
لتطابق `enabled_heads` الافتراضي في خط الأنابيب — فهي هنا أهداف تدريب
حقيقية (`task_type='classification'`) لا مجرّد مخرجات غير مُستخدَمة.

⚠️ **ترميز التصنيف**: خط الأنابيب يُخرج `y_{هدف}_class` بقيم `+1.0`/`-1.0`
(اتجاه، لا احتمال)، بينما رأس `binary_classification` (sigmoid) وخسارة
`trainer_framework` (`binary_crossentropy`) يفترضان `{0,1}` القياسية —
اكتُشف هذا فقط بالتدريب الفعلي على بيانات حقيقية (الدقّة المُبلَّغة أثناء
التدريب كانت صفراً دائماً بلا هذا التحويل). `_to_unit_label` أدناه تحوّل
`(y+1)/2` عند التغذية فقط — لا تعديل على أي دفتر آخر.

⚠️ **أعطال رُصدت في أول تدريب حقيقي (60 حقبة، بيانات 1D) وأُصلحت هنا وفي `trainer_framework`:**
* **انهيار موازنة Kendall**: خسارة NIG (NLL) تصبح سالبة، ومعها `0.5·exp(-s)·L + 0.5·s` تنحدر بلا
  حدّ — `loss` هبط من `+0.9` إلى `-42` بينما `val_*_mae` ثابت تماماً، و`BestModelTracker` أعلن «أفضل
  جديد» كل حقبة. الإصلاح في `trainer_framework` (استثناء `evidential` من Kendall + مقياس
  `val_raw_loss`)، واختيار الأفضل هنا صار على `val_raw_loss`.
* **`lambda_reg` كان صفراً** (لا جداول معرَّفة) فمنظِّم الأدلة معطَّل، و**رؤوس `conf_*` بلا أي تدرّج**
  (لا خسارة معايرة) — أُضيف الجدولان و`use_calibration_loss` في `build_target_configs`/`main_config`.


In [ ]:
def build_target_configs(price_targets):
    """يبني قسم targets لإعداد trainer_framework — رأسا انحدار وتصنيف معاً
    لكل هدف، مطابقة لِما ينتجه build_model_fn الآن افتراضياً. مفاتيح القاموس
    (مثلاً 'high_reg'/'high_class') أسماء تعسّفية لِـ trainer فقط، لا تُقرأ
    من أي مكان آخر."""
    cfg = {}
    for t in price_targets:
        cfg[f"{t}_reg"] = {
            "true_key": f"y_{t}_reg",
            "task_type": "evidential",
            "output_keys": {
                "mu": f"y_{t}", "nu": f"y_{t}_nu", "alpha": f"y_{t}_alpha",
                "beta": f"y_{t}_beta", "confidence": f"y_{t}_confidence",
            },
            # منظِّم الأدلة (lambda_reg) ومعايرة رأس الثقة (lambda_calib) — جدولاهما في main_config.
            # بلاهما: lambda_reg=0 (لا شيء يمنع تضخيم الدليل على التدريب)، ورأس conf_* لا يصله أي
            # تدرّج إطلاقاً (تحذير "Gradients do not exist for conf_*") فتبقى y_{t}_confidence
            # التي تعرضها chicks أوزاناً عشوائية غير مُدرَّبة.
            "use_calibration_loss": True,
            "lambda_reg_var": "lambda_reg",
            "lambda_calib_var": "lambda_calib",
        }
        cfg[f"{t}_class"] = {
            "true_key": f"y_{t}_class",
            "task_type": "classification",
            "binary": True,
            "output_keys": {"logits": f"y_{t}_class_logits"},
        }
    return cfg

In [ ]:
# يُعرِّف: build_config، build_training_system، GenericTrainer، ...
# ويُشغِّل تلقائياً Smoke Test (بيانات تركيبية — عدّة ثوانٍ، آمن دائماً).
%run "trainer_framework_v2.ipynb"

In [ ]:
main_config = build_config({
    "run": {
        "run_dir": "/content/drive/MyDrive/training_runs/crypto_model_v1",  # ⚠️ غيّره لمسارك
        "epochs": 60,
        "batch_size": 64,
        "train_mode": "auto",
    },
    "targets": build_target_configs(PRICE_TARGETS),
    "loss": {
        # Kendall تبقى لرؤوس التصنيف فقط؛ رؤوس evidential (NLL قد تكون سالبة) تُستثنى افتراضياً في
        # trainer_framework — معها كانت الخسارة الكلية تنهار نحو -∞ (راجع ملاحظة القسم 4 هناك).
        "use_uncertainty_weighting": True,
        "schedules": {
            # قيمتان ابتدائيتان من مثال trainer_framework نفسه — عدّلهما بعد مراقبة val_*_nig_pen.
            "lambda_reg":   {"start": 0.0, "end": 0.05, "warmup_epochs": 5, "schedule": "linear"},
            "lambda_calib": {"start": 0.0, "end": 0.1,  "warmup_epochs": 5, "schedule": "cosine"},
        },
    },
    "callbacks": {
        # val_raw_loss = مجموع خسائر المهام بأوزانها الثابتة (بلا حدود Kendall المتعلَّمة)، فلا يتحسّن
        # إلا بتحسّن حقيقي على val. val_loss يتغيّر أيضاً لأن log_var نفسها تتعلّم.
        "early_stopping": {"monitor": "val_raw_loss", "mode": "min"},
    },
})

import tensorflow as tf


def _to_unit_label(y):
    """يحوّل ترميز اتجاه خط الأنابيب (+1.0/-1.0) إلى {0,1} القياسية التي
    يفترضها binary_classification (sigmoid) وخسارة classification في
    trainer_framework (binary_crossentropy). لا تُطبَّق على أهداف الانحدار."""
    return (y + 1.0) / 2.0


def _y_for(split):
    y = {}
    for cfg in main_config["targets"].values():
        v = split["y"][cfg["true_key"]]
        y[cfg["true_key"]] = _to_unit_label(v) if cfg["task_type"] == "classification" else v
    return y


def make_shuffled_dataset(X, y_dict, batch_size, seed=None):
    """خلط **كامل** كل حقبة عبر فهارس لا عبر البيانات نفسها.

    split_data يُخرج العيّنات مرتّبة عملةً عملة ثم زمنياً، و`shuffle(4096)` السابق يخلط محلياً فقط:
    قيس على 300 عملة × 1060 عيّنة — ~13 عملة فقط في كل دفعة، والحقبة تمرّ على العملات بالترتيب (أول
    10% من الحقبة ≈ العملة رقم 15، آخر 10% ≈ رقم 284)، فينتهي كل تحقق بعد تدريب شبه حصري على آخر
    عشرات العملات. خلط الفهارس (أعداد صحيحة، بضعة ميجابايت) يعطي ~58 عملة لكل دفعة بتوزيع منتظم،
    دون نسخ X (~1.8 جيجا) داخل مخزن الخلط."""
    n, keys = len(X), list(y_dict)
    arrays = [X] + [y_dict[k] for k in keys]

    def _gather(idx):
        idx = np.sort(idx)                     # ترتيب داخل الدفعة لا يغيّر التدرّج ويُسرّع القراءة
        return tuple(np.asarray(a[idx], dtype="float32") for a in arrays)

    def _map(idx):
        outs = tf.numpy_function(_gather, [idx], [tf.float32] * len(arrays))
        outs = [tf.ensure_shape(o, (batch_size,) + tuple(a.shape[1:])) for o, a in zip(outs, arrays)]
        return outs[0], dict(zip(keys, outs[1:]))

    return (tf.data.Dataset.range(n).shuffle(n, seed=seed, reshuffle_each_iteration=True)
            .batch(batch_size, drop_remainder=True)
            .map(_map, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))


train_ds = make_shuffled_dataset(train[f"X_{MODEL_TF}"], _y_for(train), main_config["run"]["batch_size"])
val_ds = (tf.data.Dataset.from_tensor_slices((val[f"X_{MODEL_TF}"], _y_for(val)))
          .batch(main_config["run"]["batch_size"], drop_remainder=True)
          .prefetch(tf.data.AUTOTUNE))
sample_batch = next(iter(train_ds))

trainer, callbacks, initial_epoch = build_training_system(model_builder, main_config, sample_batch)
history = trainer.fit(
    train_ds, validation_data=val_ds, initial_epoch=initial_epoch,
    epochs=main_config["run"]["epochs"], callbacks=callbacks, verbose=1,
)
model = trainer.model

## ٦) الاختبار — تحويل تقسيم خط الأنابيب إلى شكل `chicks`

فجوتان يسدّهما المُحوِّل التالي (انظر شرح القسم ٠):
* تسمية `y`: `y_close_reg` (خط الأنابيب) ← `close` (اسم `TargetSpec` المجرَّد
  الذي يتوقّعه `chicks`).
* `base_params`: `[آخر_سعر, آخر_سعر]` بدل عمود `base_params` الخام في
  `dataset` — يجعل صيغة فكّ تشفير `chicks` (`raw*iqr + median`) تُكافئ
  عكس العائد المباشر (`آخر_سعر × (١ + عائد)`)، المطابق لـ`reg_target_mode='return'`.
  ⚠️ هذا صحيح لـ`close` فقط: `high`/`low` عوائد نسبة لآخر high/low لا لآخر close، و`base_params`
  مشتركة بين الأهداف — فتُفكّ عبر `EVAL_TARGET_SPECS` (`relative_to_entry=True`) التي تتجاهل
  `base_params` وتستخدم سعر دخول كل هدف من `last_candles`. **صالح فقط لهذا الوضع** — إن حوّلت `reg_target_mode` إلى `'window_scale'`
  مرّر `split['base_params']` الخام بدل هذا التحويل.

`chicks` يبقى مخصَّصاً للأهداف المستمرة (high/low/close) فقط — لا يوجد فيه
مسار تصنيف جاهز (تسمية مخرجاته `y_{name}`/`_epistemic`/`_aleatoric`/
`_confidence` لا تُطابق `y_{name}_class_logits`). تقييم رؤوس التصنيف
(المُدرَّبة فعلياً من القسم ٥) عبر `classification_accuracy_report` في
القسم ٧ بدلاً من ذلك — تقرير مستقلّ بسيط لا عبر خط أنابيب `chicks` الكامل.


In [ ]:
LAST_CLOSE_COL = LAST_COLUMNS.index("last_close")


def build_chicks_test_dict(pipeline_test, model_tf, reg_target_mode=None):
    """يحوّل `test` (مخرَج split_data، قاموس {أصل: قسم}) إلى الشكل الذي
    تتوقعه دوال chicks (`test_all_assets_v4`/`run_full_analysis`)."""
    reg_target_mode = reg_target_mode or CONFIG.get("reg_target_mode", "return")
    out = {}
    for asset, split in pipeline_test.items():
        last_candles = split["last_candles"]
        if reg_target_mode == "return":
            last_close = last_candles[:, LAST_CLOSE_COL]
            base_params_eval = np.stack([last_close, last_close], axis=1).astype("float32")
        else:
            base_params_eval = split["base_params"]
        out[asset] = {
            f"X_{model_tf}": split[f"X_{model_tf}"],
            "base_params": base_params_eval,
            "last_candles": last_candles,
            "y": {t: split["y"][f"y_{t}_reg"] for t in PRICE_TARGETS},
        }
    return out


import numpy as np

test_dict = build_chicks_test_dict(test, MODEL_TF)

In [ ]:
# يُعرِّف: TargetSpec، DEFAULT_PRICE_TARGETS، predict_with_evaluation_v4،
# test_all_assets_v4، predict_latest_v4/predict_latest_all_assets،
# run_full_analysis، ...
%run "chicks_v4_5_input_output_patterns.ipynb"

import dataclasses
# reg_target_mode='return': كل هدف عائد نسبة لآخر سعر من *نفس نوعه* (high لآخر high، low لآخر low).
# base_params واحدة مشتركة بين الأهداف لا تستطيع تمثيل ذلك — كانت high/low تُفكّ حول آخر close
# فتنزاح أسعارها وتنحرف كل مقاييسها (MAE، نسبة النجاح، جداول الحركة). relative_to_entry يفكّ كل هدف
# بـ آخر_سعره × (1 + العائد) — مطابق لـ invert_reg_predictions في خط الأنابيب.
RETURN_PRICE_TARGETS = [dataclasses.replace(s, relative_to_entry=True) for s in DEFAULT_PRICE_TARGETS]
EVAL_TARGET_SPECS = (RETURN_PRICE_TARGETS if CONFIG.get("reg_target_mode", "return") == "return"
                     else DEFAULT_PRICE_TARGETS)

In [ ]:
full_results = run_full_analysis(
    model=model,
    test_dict=test_dict,
    timeframes=[MODEL_TF],
    target_specs=EVAL_TARGET_SPECS,
    out_dir="analysis_outputs",
)
full_results["per_asset_results"]

## ٧) دوال فحص وتحقّق إضافية (تُستدعى عند الحاجة، لا تلقائياً)

In [ ]:
def latest_trading_report(n_display=5):
    """تقرير 'آخر N عيّنات' لكل أصل — للتداول الحيّ، يعمل بلا أهداف حقيقية."""
    return predict_latest_all_assets(
        model, test_dict, timeframes=[MODEL_TF], target_specs=EVAL_TARGET_SPECS,
        n_display=n_display,
    )


def real_price_predictions(asset, target):
    """يحوّل مخرَج النموذج (عائد مباشر) لسعر حقيقي عبر invert_reg_predictions
    نفسها المستخدَمة في خط الأنابيب — مفيد حين تحتاج سعراً لا عائداً.
    ``invert_reg_predictions`` تتوقّع اسم *رأس* (مثلاً 'close_reg')، لا اسم
    الهدف المجرَّد — راجع split_head_name في خط الأنابيب."""
    split = test[asset]
    x = split[f"X_{MODEL_TF}"]
    preds = model(x, training=False)[f"y_{target}"].numpy().ravel()
    return invert_reg_predictions(preds, f"{target}_reg", last_candles=split["last_candles"], config=CONFIG)


def classification_accuracy_report():
    """دقة/AUC رؤوس التصنيف الثنائي (صعود/هبوط) لكل هدف وأصل — منفصل عن
    تقرير chicks (مخصَّص للأهداف المستمرة فقط، انظر ملاحظة القسم ٦).
    y_true من خط الأنابيب بترميز +1.0/-1.0 — يُحوَّل هنا بنفس _to_unit_label
    المستخدَمة في التدريب (القسم ٥) قبل المقارنة."""
    from sklearn.metrics import accuracy_score, roc_auc_score
    rows = []
    for asset, split in test.items():
        x = split[f"X_{MODEL_TF}"]
        out = model(x, training=False)
        for t in PRICE_TARGETS:
            y_true = _to_unit_label(np.asarray(split["y"][f"y_{t}_class"]).ravel())
            y_prob = out[f"y_{t}_class_logits"].numpy().ravel()
            y_pred = (y_prob >= 0.5).astype("float32")
            auc = roc_auc_score(y_true, y_prob) if len(set(y_true.tolist())) > 1 else float("nan")
            rows.append({"asset": asset, "target": t, "n": len(y_true),
                         "accuracy": accuracy_score(y_true, y_pred), "auc": auc})
    return pd.DataFrame(rows)

In [ ]:
def _pool_by_asset(per_asset_arrays: dict, order=None):
    """يدمج {أصل: مصفوفة} في مصفوفة واحدة + قائمة asset_bounds (نفس تنسيق
    `rolling_splits`/`build_dataset_live` في خط الأنابيب) — أساس بسيط لدمج
    عدّة أصول في دفعة واحدة."""
    order = order or list(per_asset_arrays.keys())
    parts, bounds, pos = [], [], 0
    for name in order:
        arr = np.asarray(per_asset_arrays[name])
        n = len(arr)
        bounds.append({"name": name, "start": pos, "end": pos + n})
        parts.append(arr)
        pos += n
    return np.concatenate(parts, axis=0), bounds


def pool_test_dict(test_dict, model_tf, n_per_asset=None):
    """يحوّل `test_dict` ({أصل: قسم}، شكل chicks) إلى **دفعة واحدة مدمجة**
    (X/base_params/last_candles + `asset_bounds`) — بدل تشغيل النموذج على
    كل أصل على حدة. `n_per_asset`: خذ آخر N عيّنة فقط لكل أصل (مثلاً 5 أو
    10) قبل الدمج — مفيد لاختبار سريع بدفعة واحدة بدل حلقة يدوية لكل عملة.

    يُرجع ``(pooled, y_true_pooled)``: ``pooled`` جاهز لـ
    `predict_pooled_batch_by_asset` مباشرة، و``y_true_pooled`` بمفاتيح
    ``y_{هدف}`` (إن وُجدت أهداف فعلية) لتمريره اختيارياً لنفس الدالة."""
    order = list(test_dict.keys())
    x_by_asset, bp_by_asset, lc_by_asset, y_by_target = {}, {}, {}, {}
    for name, split in test_dict.items():
        n = len(split["base_params"])
        k = n if n_per_asset is None else min(n_per_asset, n)
        sl = slice(n - k, n)
        x_by_asset[name] = split[f"X_{model_tf}"][sl]
        bp_by_asset[name] = split["base_params"][sl]
        if split.get("last_candles") is not None:
            lc_by_asset[name] = split["last_candles"][sl]
        for t, arr in (split.get("y") or {}).items():
            y_by_target.setdefault(t, {})[name] = np.asarray(arr)[sl]

    X_pooled, asset_bounds = _pool_by_asset(x_by_asset, order)
    base_params_pooled, _ = _pool_by_asset(bp_by_asset, order)
    last_candles_pooled = _pool_by_asset(lc_by_asset, order)[0] if lc_by_asset else None
    y_true_pooled = {f"y_{t}": _pool_by_asset(d, order)[0] for t, d in y_by_target.items()}
    pooled = {f"X_{model_tf}": X_pooled, "base_params": base_params_pooled,
              "last_candles": last_candles_pooled, "asset_bounds": asset_bounds}
    return pooled, y_true_pooled


def predict_pooled_batch_by_asset(model, pooled, model_tf, target_specs=None, n_display=5,
                                  y_true_pooled=None, timestamp_col=None, verbose=True):
    """يُشغِّل النموذج **مرّة واحدة فقط** على دفعة مدمجة من عدّة أصول معاً
    (``pooled`` بتنسيق `asset_bounds` — من `pool_test_dict` أعلاه، أو مباشرة
    من `extract_last_batch(build_dataset_live(...), n)` في خط الأنابيب
    للتداول الحيّ)، ثم يقسّم النتائج حسب كل أصل عبر `asset_bounds` ليبني
    تقريراً واحداً — بدل استدعاء النموذج مرّة لكل أصل كما في
    `predict_latest_all_assets`/`test_all_assets_v4`. فكّ التشفير والتجميع
    (`decode_predictions_v4`/`build_latest_table`) رخيصان (numpy فقط، بلا
    نموذج) فيُعادان لكل قسم أصل بأمان بلا كلفة إضافية تُذكَر.

    مثال تداول حيّ مباشر (بلا `test_dict` إطلاقاً):
        live_ds = build_dataset_live(["BTCUSDT", "ETHUSDT", "SOLUSDT"])
        live_batch = extract_last_batch(live_ds, n=1)   # آخر شمعة لكل عملة
        predict_pooled_batch_by_asset(model, live_batch, MODEL_TF,
                                      target_specs=EVAL_TARGET_SPECS, n_display=1)
    """
    X_inputs = (pooled[f"X_{model_tf}"],)
    specs = _resolve_for_model(target_specs, model, X_inputs)
    raw = predict_batch_v4(model, X_inputs, specs)  # ← استدعاء نموذج واحد فقط لكل الأصول معاً

    tables = []
    for b in pooled["asset_bounds"]:
        s, e = b["start"], b["end"]
        raw_slice = {k: v[s:e] for k, v in raw.items()}
        bp_slice = pooled["base_params"][s:e] if pooled.get("base_params") is not None else None
        lc_slice = pooled["last_candles"][s:e] if pooled.get("last_candles") is not None else None
        decoded = decode_predictions_v4(raw_slice, specs, bp_slice, lc_slice)
        y_true = ({k: v[s:e] for k, v in y_true_pooled.items()} if y_true_pooled else None)
        if y_true:
            decoded = evaluate_predictions_v4(decoded, specs, y_true, bp_slice)
        k_disp = min(n_display, e - s) if n_display else (e - s)
        table = build_latest_table(decoded, specs, n_display=k_disp,
                                   timestamps=_extract_timestamps(None, lc_slice, timestamp_col),
                                   asset=b["name"])
        tables.append(table)

    result = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    if verbose:
        print(f"\n🚀 دُفعة واحدة مدمجة ({len(pooled['asset_bounds'])} أصل، استدعاء نموذج واحد فقط):")
        print_latest_table(result, legend=True)
    return result

## ٧-ب) تقييم انتقائي: دقة ≥ 65% على شريحة واثقة، أو صفقات 2:1

الدقة الإجمالية ليست المعيار هنا: يكفي أن تكون **الثقة متّسقة** — أن ترتفع الدقة فعلاً كلما
ارتفعت الثقة — فنتداول الشريحة الواثقة فقط. هذا القسم يقيس ذلك بصرامة:

* **`selective_direction_report`**: لكل مصدر ثقة (`class_margin` = |احتمال الصعود − 0.5|،
  `nig_edge` = |العائد المتوقَّع| ÷ عرض Student-t، `conf_head` = رأس الثقة، `agree_margin` =
  هامش التصنيف حين يتّفق مع اتجاه الانحدار) يحسب اتساق الدقة عبر عُشيرات الثقة (سبيرمان،
  +1 = اتساق تام)، ويختار **على val فقط** أوسع شريحة بلغت الدقة المطلوبة، ثم يقيسها على test.
* **`rr_trading_report`**: صفقة بدخول عند إغلاق اليوم، وقف على مسافة `d`، وهدف على `2·d`
  (`d` = عرض Student-t للإغلاق، أو المسافة للقاع/القمة المتوقَّعة مع `stop_mode="model_levels"`).
  تُقيَّم بقمة وقاع وإغلاق الغد الفعلية بعد التكلفة، ويُقارَن الناتج بنفس الصفقات **باتجاه عشوائي**.

⚠️ ثلاث حقائق يبنى عليها الحكم:
1. **العتبة من val والحكم من test** — شريحة تبلغ 65% على val وحده قد تكون صدفة (اختيرت منه).
2. **ربح 40% عند 2:1 ليس سقفاً منخفضاً**: في سعر عشوائي يُلمس هدف 2R قبل وقف 1R بنحو ثلث
   المرات، فنقطة التعادل ≈ 33% + التكلفة. خط الأساس العشوائي في التقرير يُظهر هذا الرقم فعلياً.
3. **الشمعة اليومية لا تخبر أيّهما لُمس أولاً**: لمس الهدف والوقف معاً يُحسب خسارة (`both_hit`).
   والعملات المترابطة في نفس اليوم ليست عيّنات مستقلة — انظر `test_dates` لا `test_n` وحده.

### 🗂️ مؤجَّل للبناء لاحقاً: «مُقيِّم ثقة» يُدرَّب على بيانات لم يرها النموذج

**لماذا:** كل مصادر الثقة أعلاه تعلّمت على `train`، حيث يحفظ النموذج العيّنات (في أول تدريب
حقيقي: دقة `low_class` 88% على train مقابل 68% على val). رأس ثقة أو احتمال تصنيف يتعلّم هناك
يرى عالماً شبه خالٍ من الأخطاء، فلا يتعلّم كيف يبدو الخطأ. ورأس `conf_*` الحالي مُدرَّب على
*حجم* خطأ الانحدار (يتبع التقلّب)، لا على *صحة الاتجاه* التي يحتاجها التداول.

**الفكرة (بروح ConfidNet، Corbière et al. 2019، لكن على بيانات محجوزة):** نموذج صغير منفصل
يتنبأ «هل توقّع النموذج المجمَّد لهذه العيّنة صحيح؟»، ويتعلّم من أخطاء حقيقية خارج التدريب.

**التصميم:**
1. **تقسيم زمني ثلاثي بلا خلط:** `val` يُقسَم زمنياً إلى `val_A` (الأقدم) ثم `val_B`، بفجوة
   عزل بينهما (`window + forecast_horizon` شمعة، كما في `split_data`)، ثم `test` كما هو.
   * `val_A`: التوقف المبكر للنموذج الرئيسي + تدريب المُقيِّم.
   * `val_B`: اختيار عتبة المُقيِّم فقط (`pick_threshold`).
   * `test`: الحكم النهائي فقط — لا يُعاد التكرار عليه بعد رؤية نتيجته.
2. **الهدف:** `correct = 1[اتجاه التوقع == اتجاه إغلاق الغد]` لدقة الاتجاه؛ أو `1[R > 0]` من
   `simulate_rr_trades` لصفقات 2:1 (مُقيِّم منفصل لكل هدف).
3. **المدخلات — من مخرجات النموذج المجمَّد فقط** (لا نوافذ الميزات الخام، كي لا يُعاد الحفظ):
   `p_up_close`، `|p−0.5|`، `mu_close`، `wst_close`، `|mu|/wst`، `nu`، `alpha`، اتفاق اتجاه
   التصنيف والانحدار، نظيراتها لـhigh/low، وتقلّب حديث واحد (آخر قيمة `NATR_14`). كلها موجودة
   في `collect_signals` عدا الأخيرة.
4. **النموذج:** انحدار لوجستي بتنظيم قوي أولاً (قليل التباين، ترتيبه مفهوم)؛ ثم شجرة تعزيز
   ضحلة (عمق 2-3) فقط إن تفوّقت على `val_B`. بيانات val صغيرة (بضع مئات من الأيام الفعلية على
   فريم يومي) فالبساطة شرط لا خيار.
5. **التقييم:** مصدر ثقة خامس `heldout_selector` داخل `direction_scores`/`rr_trading_report`
   بنفس القواعد تماماً. يُعتمد فقط إن تفوّق على **أفضل مصدر خام على test**، لا على val.
6. **محاذير:** عيّنات العملات في نفس اليوم مترابطة — أي تحقّق متقاطع داخل `val_A` يُجمَّع
   بالتاريخ (`timestamp`) لا بالعيّنة؛ وأي معايرة لاحتمال المُقيِّم (isotonic) تُجرى على `val_B`
   لا على `val_A`.

**أين يُبنى:** دوال بجوار هذا القسم — `split_val_chronologically(val, frac=0.5)`،
`fit_confidence_selector(val_a_df, target)`، `selector_scores(df, selector)` — بلا أي تعديل على
النموذج أو `trainer_framework`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 تقييم انتقائي: هل ترتفع الدقة فعلاً مع الثقة؟ وهل تصمد صفقات 2:1؟
# ═══════════════════════════════════════════════════════════════════════════
# المبدأ: كل عتبة تُختار على val فقط، ثم تُقاس على test الذي لم يُلمس. الثقة
# «متّسقة» فقط إن كانت دقة الشريحة الأعلى ثقة أعلى من الأدنى على test أيضاً —
# لا يكفي أن يبدو ذلك على val (العتبة نفسها اختيرت منه).
import numpy as np
import pandas as pd

_LC = {name: i for i, name in enumerate(LAST_COLUMNS)}


def _concat_splits(split_or_dict, model_tf):
    """split واحد (train/val) أو قاموس {أصل: split} (test) ← split واحد + عمود asset."""
    if "y" in split_or_dict:
        n = len(split_or_dict["last_candles"])
        return split_or_dict, np.array(["all"] * n)
    parts = list(split_or_dict.items())
    merged = {
        f"X_{model_tf}": np.concatenate([s[f"X_{model_tf}"] for _, s in parts]),
        "last_candles": np.concatenate([s["last_candles"] for _, s in parts]),
        "y": {k: np.concatenate([s["y"][k] for _, s in parts]) for k in parts[0][1]["y"]},
    }
    assets = np.concatenate([np.array([a] * len(s["last_candles"])) for a, s in parts])
    return merged, assets


def collect_signals(model, split_or_dict, model_tf, batch_size=1024, outputs=None):
    """مخرجات النموذج + الحقيقة الفعلية للشمعة التالية في DataFrame واحد (صف لكل عيّنة).
    ``outputs``: مخرجات جاهزة (dict) بدل استدعاء النموذج — للاختبار فقط."""
    split, assets = _concat_splits(split_or_dict, model_tf)
    if outputs is None:
        outputs = model.predict(split[f"X_{model_tf}"], batch_size=batch_size, verbose=0)
    o = {k: np.asarray(v, dtype="float64").reshape(len(assets), -1)[:, 0] for k, v in outputs.items()}
    lc = np.asarray(split["last_candles"], dtype="float64")
    df = pd.DataFrame({
        "asset": assets,
        "timestamp": lc[:, _LC["timestamp"]],
        "entry": lc[:, _LC["last_close"]],
        "last_high": lc[:, _LC["last_high"]], "last_low": lc[:, _LC["last_low"]],
        "fut_close": lc[:, _LC["future_close"]],
        "fut_high": lc[:, _LC["future_high_max"]], "fut_low": lc[:, _LC["future_low_min"]],
    })
    for t in PRICE_TARGETS:
        if f"y_{t}" not in o:
            continue
        nu, alpha, beta = o[f"y_{t}_nu"], o[f"y_{t}_alpha"], o[f"y_{t}_beta"]
        df[f"mu_{t}"] = o[f"y_{t}"]
        df[f"wst_{t}"] = np.sqrt(beta * (1.0 + nu) / (nu * alpha))   # عرض Student-t (بوحدات العائد)
        if f"y_{t}_confidence" in o:
            df[f"conf_{t}"] = o[f"y_{t}_confidence"]
        if f"y_{t}_class_logits" in o:
            df[f"p_up_{t}"] = o[f"y_{t}_class_logits"]           # احتمال بعد sigmoid
    df["pred_high"] = df["last_high"] * (1.0 + df["mu_high"]) if "mu_high" in df else np.nan
    df["pred_low"] = df["last_low"] * (1.0 + df["mu_low"]) if "mu_low" in df else np.nan
    df["up"] = (df["fut_close"] > df["entry"]).astype(int)
    return df


def direction_scores(df, target="close"):
    """مرشّحات (اتجاه، درجة ثقة) — الأعلى درجة = الأكثر ثقة. كلها تتنبأ باتجاه الإغلاق بعد الأفق."""
    out = {}
    if f"p_up_{target}" in df:
        p = df[f"p_up_{target}"].to_numpy()
        out["class_margin"] = (np.where(p >= 0.5, 1, -1), np.abs(p - 0.5))
    if f"mu_{target}" in df:
        mu, wst = df[f"mu_{target}"].to_numpy(), df[f"wst_{target}"].to_numpy()
        side = np.where(mu >= 0, 1, -1)
        out["nig_edge"] = (side, np.abs(mu) / (wst + 1e-12))       # |تحرك متوقع| / عدم اليقين
        if f"conf_{target}" in df:
            out["conf_head"] = (side, df[f"conf_{target}"].to_numpy())
        if "class_margin" in out:
            agree = out["class_margin"][0] == side
            out["agree_margin"] = (side, np.where(agree, out["class_margin"][1], -np.inf))
    return out


def wilson_interval(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def selective_curve(score, correct, n_bins=10):
    """دقة كل شريحة ثقة (عُشر) + ارتباط سبيرمان بين رتبة الشريحة ودقتها (1.0 = اتساق تام)."""
    ok = np.isfinite(score)
    s, c = score[ok], correct[ok].astype(float)
    if len(s) < n_bins:
        return pd.DataFrame(), np.nan
    edges = np.unique(np.quantile(s, np.linspace(0, 1, n_bins + 1)))
    b = np.clip(np.searchsorted(edges, s, side="right") - 1, 0, len(edges) - 2)
    rows = [{"bin": i + 1, "n": int((b == i).sum()), "accuracy": c[b == i].mean()}
            for i in range(len(edges) - 1) if (b == i).any()]
    table = pd.DataFrame(rows)
    rho = table["bin"].corr(table["accuracy"], method="spearman") if len(table) > 2 else np.nan
    return table, rho


def pick_threshold(score, correct, target_acc=0.65, min_n=200):
    """أكبر تغطية على val تبلغ فيها دقة الشريحة الأعلى ثقة target_acc (مع ≥ min_n عيّنة).
    يُرجع العتبة أو None إن لم تبلغ أي شريحة الهدف."""
    ok = np.isfinite(score)
    s, c = score[ok], correct[ok].astype(float)
    order = np.argsort(-s, kind="stable")
    cum_acc = np.cumsum(c[order]) / np.arange(1, len(s) + 1)
    k_ok = np.where((cum_acc >= target_acc) & (np.arange(1, len(s) + 1) >= min_n))[0]
    return None if len(k_ok) == 0 else float(s[order][k_ok[-1]])


def _subset_stats(df, mask, correct):
    n, k = int(mask.sum()), int(correct[mask].sum())
    lo, hi = wilson_interval(k, n)
    return {"n": n, "n_dates": int(df.loc[mask, "timestamp"].nunique()),
            "coverage": n / max(len(df), 1), "accuracy": k / n if n else np.nan,
            "wilson_lo": lo, "wilson_hi": hi}


def selective_direction_report(val_df, test_df, target_acc=0.65, min_n=200, target="close", verbose=True):
    """لكل مرشّح ثقة: اتساق (دقة ترتفع مع الثقة؟) على val وtest، ثم عتبة من val تُقاس على test."""
    v_scores, t_scores = direction_scores(val_df, target), direction_scores(test_df, target)
    rows = []
    for name in v_scores:
        v_side, v_s = v_scores[name]
        t_side, t_s = t_scores[name]
        v_ok = (v_side == np.where(val_df["up"] == 1, 1, -1)).astype(int)
        t_ok = (t_side == np.where(test_df["up"] == 1, 1, -1)).astype(int)
        _, rho_v = selective_curve(v_s, v_ok)
        _, rho_t = selective_curve(t_s, t_ok)
        thr = pick_threshold(v_s, v_ok, target_acc, min_n)
        row = {"score": name, "acc_all_test": t_ok.mean(), "monotonic_val": rho_v, "monotonic_test": rho_t,
               "threshold": thr}
        if thr is not None:
            vs = _subset_stats(val_df, v_s >= thr, v_ok)
            ts = _subset_stats(test_df, t_s >= thr, t_ok)
            row.update({"val_acc": vs["accuracy"], "val_n": vs["n"],
                        "test_acc": ts["accuracy"], "test_n": ts["n"], "test_dates": ts["n_dates"],
                        "test_coverage": ts["coverage"], "test_wilson_lo": ts["wilson_lo"],
                        "test_wilson_hi": ts["wilson_hi"]})
            passed = ts["n"] >= min_n and ts["accuracy"] >= target_acc and ts["wilson_lo"] > 0.5
            row["verdict"] = "✅ صمد على test" if passed else "❌ لم يصمد على test"
        else:
            row["verdict"] = f"❌ لا شريحة على val بلغت {target_acc:.0%}"
        rows.append(row)
    report = pd.DataFrame(rows)
    if verbose:
        print(f"\n🎯 دقة الاتجاه الانتقائية (الهدف ≥ {target_acc:.0%} على test، العتبة من val فقط)")
        print("   monotonic_* = سبيرمان بين شريحة الثقة ودقتها: قرب +1 اتساق، قرب 0 الثقة لا تعني شيئاً")
        print("   ⚠️ فاصل Wilson يفترض عيّنات مستقلة؛ العملات المترابطة في نفس اليوم ليست كذلك —"
              " انظر test_dates (عدد الأيام الفعلية) لا test_n وحده")
        with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
            print(report.to_string(index=False))
    return report


def simulate_rr_trades(df, side, rr=2.0, stop_mode="nig", stop_k=1.0, cost_pct=0.08):
    """صفقة لكل صف: دخول عند last_close، وقف على مسافة d، هدف على rr·d. يُقيَّم بقمة/قاع/إغلاق
    الأفق الفعلي (forecast_horizon شمعة؛ يوم واحد في الإعداد اليومي).

    stop_mode:
      'nig'          → d = stop_k × عرض Student-t لـ close (عدم يقين النموذج نفسه)
      'model_levels' → d = المسافة للقاع المتوقَّع (شراء) / للقمة المتوقَّعة (بيع)
    ⚠️ شمعة يومية لا تخبر أيّهما لُمس أولاً: لمس الهدف والوقف معاً يُحسب خسارة (افتراض محافظ).
    بلا لمس لأيّهما: خروج عند إغلاق الغد. cost_pct: تكلفة الذهاب والإياب % (افتراضي عمولة+انزلاق
    خط الأنابيب 0.06+0.02). يُرجع R لكل صفقة (بعد التكلفة) ونوع الخروج."""
    entry = df["entry"].to_numpy()
    if stop_mode == "nig":
        d = stop_k * df["wst_close"].to_numpy()
    elif stop_mode == "model_levels":
        d = np.where(side > 0, (entry - df["pred_low"].to_numpy()) / entry,
                     (df["pred_high"].to_numpy() - entry) / entry)
    else:
        raise ValueError("stop_mode: 'nig' | 'model_levels'")
    valid = np.isfinite(d) & (d > 1e-5)
    d = np.where(valid, d, np.nan)
    tp = entry * (1 + side * rr * d)
    sl = entry * (1 - side * d)
    hi, lo, cl = df["fut_high"].to_numpy(), df["fut_low"].to_numpy(), df["fut_close"].to_numpy()
    hit_tp = np.where(side > 0, hi >= tp, lo <= tp)
    hit_sl = np.where(side > 0, lo <= sl, hi >= sl)
    r = np.where(hit_sl, -1.0, np.where(hit_tp, rr, side * (cl - entry) / entry / d))
    r = r - (cost_pct / 100.0) / d
    exit_kind = np.where(hit_sl & hit_tp, "both", np.where(hit_sl, "sl", np.where(hit_tp, "tp", "close")))
    return pd.DataFrame({"r": np.where(valid, r, np.nan), "exit": np.where(valid, exit_kind, "invalid"),
                         "timestamp": df["timestamp"].to_numpy()})


def _trade_stats(tr):
    t = tr[tr["exit"] != "invalid"]
    n = len(t)
    if n == 0:
        return {"n": 0}
    # t-stat على مستوى الأيام: صفقات نفس اليوم مترابطة (العملات تتحرك معاً)، فالعيّنة المستقلة هي
    # اليوم لا الصفقة — متوسط R لكل يوم، ثم متوسط/خطأ معياري عبر الأيام.
    daily = t.groupby("timestamp")["r"].mean()
    t_days = float(daily.mean() / (daily.std(ddof=1) / np.sqrt(len(daily)))) if len(daily) > 2 and daily.std() > 0 else 0.0
    return {"n": n, "n_dates": int(t["timestamp"].nunique()),
            "win_rate": (t["exit"] == "tp").mean(), "both_hit": (t["exit"] == "both").mean(),
            "expectancy_R": t["r"].mean(), "total_R": t["r"].sum(), "t_days": t_days}


def rr_trading_report(val_df, test_df, rr=2.0, stop_mode="nig", stop_k=1.0, cost_pct=0.08,
                      min_trades=200, quantiles=(0.5, 0.7, 0.8, 0.9, 0.95), target="close",
                      seed=0, verbose=True):
    """لكل مرشّح ثقة: عتبة تُختار على val (أعلى متوسط R صافٍ بين عدّة شرائح)، ثم تُقاس على test.
    خط الأساس: نفس المستويات والعيّنات باتجاه عشوائي — الفرق عنه هو الميزة الفعلية.
    النجاح يتطلّب: ربحاً على val (وإلا فالعتبة المختارة لم تجد ميزة)، وربحاً على test بدلالة t ≥ 2
    محسوبة على متوسطات الأيام (test_t_days) — لا على عدد الصفقات، لأن صفقات اليوم الواحد مترابطة.
    (نسبة الربح النظرية لمشي عشوائي عند rr=2 ≈ 1/3، فالربح 40% لا يكفي وحده دليلاً.)"""
    rng = np.random.default_rng(seed)
    v_scores, t_scores = direction_scores(val_df, target), direction_scores(test_df, target)
    rows = []
    for name in v_scores:
        v_side, v_s = v_scores[name]
        t_side, t_s = t_scores[name]
        v_tr = simulate_rr_trades(val_df, v_side, rr, stop_mode, stop_k, cost_pct)
        best = None
        for q in quantiles:
            thr = float(np.nanquantile(v_s[np.isfinite(v_s)], q))
            st = _trade_stats(v_tr[v_s >= thr])
            if st["n"] >= min_trades and (best is None or st["expectancy_R"] > best[1]["expectancy_R"]):
                best = (thr, st)
        if best is None:
            rows.append({"score": name, "verdict": f"❌ أقل من {min_trades} صفقة على val"})
            continue
        thr, vst = best
        sel = t_s >= thr
        tst = _trade_stats(simulate_rr_trades(test_df[sel], t_side[sel], rr, stop_mode, stop_k, cost_pct))
        rand_side = rng.choice([-1, 1], size=int(sel.sum()))
        base = _trade_stats(simulate_rr_trades(test_df[sel], rand_side, rr, stop_mode, stop_k, cost_pct))
        # ✅ يتطلّب ربحاً على val أيضاً (العتبة اختيرت منه — إن كانت أفضل شريحة خاسرة على val فلا ميزة
        # اكتُشفت، وأي ربح على test صدفة) + دلالة على مستوى الأيام (t ≥ 2) لا على عدد الصفقات.
        passed = tst.get("n", 0) >= min_trades and vst["expectancy_R"] > 0 and tst["expectancy_R"] > 0 \
            and tst.get("t_days", 0.0) >= 2.0 and tst["expectancy_R"] > base.get("expectancy_R", np.inf)
        rows.append({"score": name, "threshold": thr,
                     "val_n": vst["n"], "val_win": vst["win_rate"], "val_exp_R": vst["expectancy_R"],
                     "test_n": tst.get("n", 0), "test_dates": tst.get("n_dates", 0),
                     "test_win": tst.get("win_rate"), "test_both_hit": tst.get("both_hit"),
                     "test_exp_R": tst.get("expectancy_R"), "test_t_days": tst.get("t_days"),
                     "random_dir_win": base.get("win_rate"), "random_dir_exp_R": base.get("expectancy_R"),
                     "verdict": ("✅ ربح على val وtest، دالّ على مستوى الأيام، وأفضل من اتجاه عشوائي" if passed
                                 else "❌ لم يصمد (يلزم: val_exp_R>0 و test_exp_R>0 و test_t_days≥2)")})
    report = pd.DataFrame(rows)
    if verbose:
        print(f"\n💹 صفقات {rr:g}:1 (وقف={stop_mode}, تكلفة {cost_pct}% ذهاباً وإياباً، العتبة من val فقط)")
        print("   win = نسبة بلوغ الهدف فعلاً؛ exp_R = متوسط الربح بوحدات المخاطرة بعد التكلفة (> 0 = مربح)؛ both_hit = لمس الهدف والوقف معاً، حُسب خسارة")
        with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 220):
            print(report.to_string(index=False))
    return report


def selective_evaluation(model=None, val_split=None, test_split=None, model_tf=None, target_acc=0.65,
                         rr=2.0, stop_mode="nig", min_n=200, val_df=None, test_df=None, verbose=True):
    """نقطة دخول واحدة: يجمع إشارات val/test ثم يُخرج التقريرين. مرّر val_df/test_df جاهزين لتفادي
    إعادة التنبؤ عند تجربة إعدادات صفقات مختلفة."""
    model_tf = model_tf or MODEL_TF
    val_df = val_df if val_df is not None else collect_signals(model, val_split, model_tf)
    test_df = test_df if test_df is not None else collect_signals(model, test_split, model_tf)
    return {
        "val_df": val_df, "test_df": test_df,
        "direction": selective_direction_report(val_df, test_df, target_acc=target_acc, min_n=min_n,
                                                verbose=verbose),
        "trades": rr_trading_report(val_df, test_df, rr=rr, stop_mode=stop_mode, min_trades=min_n,
                                    verbose=verbose),
    }


# الاستخدام (بعد التدريب):
#   sel = selective_evaluation(model, val, test)
#   rr_trading_report(sel["val_df"], sel["test_df"], rr=2.0, stop_mode="model_levels")   # بلا إعادة تنبؤ

## ٧-ج) هل يضيف النموذج شيئاً فوق شكل الشمعة؟ (رأسا high/low)

`high_class` = «هل يكسر الغد قمة اليوم؟» و`low_class` = «هل يبقى قاع الغد فوق قاع اليوم؟». النموذج
يتفوّق فيهما على نسبة الفئة الأكبر بـ14-16 نقطة، لكن جزءاً كبيراً من الجواب مكتوب في الشمعة الأخيرة
وحدها: إغلاق قرب القمة يعني أن الغد يكسرها غالباً. `candle_baseline_report` يبني خطوط أساس لا ترى
إلا ذلك، كلها مُدرَّبة على `train` فقط:

| خط الأساس | ما يراه |
|---|---|
| `rule_cpos` | موضع الإغلاق داخل مدى الشمعة `(close−low)/(high−low)` وحده |
| `candle_gbm` | شجرة تعزيز ضحلة على 4 خصائص خام للشمعة الأخيرة (الموضع، الفتيلان، المدى) |
| `laststep_gbm` | ميزات النموذج نفسها لآخر يوم فقط — إن عادلت النموذج، فتاريخ النافذة لا يضيف شيئاً |

**الحكم على test** بـAUC (بلا عتبة)، وبفاصل ثقة bootstrap يُعيد سحب **الأيام** لا العيّنات:
* ✅ النموذج يضيف معلومة: دمج درجته مع أفضل خط أساس (الدمج يُتعلَّم على val) يرفع AUC على test بفاصل كله فوق الصفر.
* ❌ القاعدة تعادله أو تتفوّق عليه — النموذج لا يضيف شيئاً فوق شكل الشمعة.
* ➖ لا فرق دالّ.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🕯️ هل يضيف النموذج شيئاً فوق شكل الشمعة؟ (high_class / low_class)
# ═══════════════════════════════════════════════════════════════════════════
# high_class = «هل يكسر الغد قمة اليوم؟» (future_high > last_high)، low_class = «هل يبقى قاع
# الغد فوق قاع اليوم؟» (future_low > last_low). كلاهما يُقرأ جزئياً من الشمعة الأخيرة وحدها:
# إغلاق قرب القمة → الغد يكسرها غالباً. هنا نبني خطوط أساس لا ترى إلا ذلك، ونقارن بالنموذج.
#
#   rule_cpos     : موضع الإغلاق داخل مدى الشمعة (close−low)/(high−low) وحده (انحدار لوجستي).
#   candle_gbm    : شجرة تعزيز ضحلة على 4 خصائص خام للشمعة الأخيرة (الموضع، الفتيلان، المدى).
#   laststep_gbm  : شجرة تعزيز على كل ميزات النموذج نفسها لكن لآخر يوم فقط (X[:, -1, :]) —
#                   إن عادلته، فتاريخ الـ32 يوماً الذي يقرؤه المحوّل لا يضيف شيئاً.
# كل خطوط الأساس تُدرَّب على train فقط، وتُقارَن على val وtest.
# الحكم على test: AUC (بلا عتبة)، وفرق AUC بفاصل ثقة bootstrap مُجمَّع بالأيام (صفقات اليوم مترابطة).
# «القيمة المضافة»: انحدار لوجستي على val بدرجة أفضل خط أساس ± درجة النموذج — هل تحسّن test؟
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score


def _candle_frame(split_or_dict, model_tf):
    split, assets = _concat_splits(split_or_dict, model_tf)
    lc = np.asarray(split["last_candles"], dtype="float64")
    h, l, c = lc[:, _LC["last_high"]], lc[:, _LC["last_low"]], lc[:, _LC["last_close"]]
    rng = np.maximum(h - l, 1e-12)
    df = pd.DataFrame({
        "asset": assets, "timestamp": lc[:, _LC["timestamp"]],
        "cpos": (c - l) / rng,                        # 1 = أغلق عند القمة، 0 = عند القاع
        "upper_wick": (h - c) / c, "lower_wick": (c - l) / c, "range_rel": (h - l) / c,
    })
    for t in ("high", "low"):
        y = np.asarray(split["y"][f"y_{t}_class"]).ravel()
        df[f"y_{t}"] = (y > 0).astype(int)             # ترميز خط الأنابيب ±1 → {0,1}
    # فحص تعريف الهدف من الأسعار الخام نفسها (يجب أن يطابق ~100%)
    df.attrs["label_check"] = {
        "high": float(np.mean(df["y_high"].values == (lc[:, _LC["future_high_max"]] > h))),
        "low": float(np.mean(df["y_low"].values == (lc[:, _LC["future_low_min"]] > l))),
    }
    return df, split


_CANDLE_COLS = ["cpos", "upper_wick", "lower_wick", "range_rel"]


def _day_bootstrap_auc_diff(y, s_a, s_b, days, n_boot=300, seed=0):
    """فرق AUC (a − b) مع فاصل 95% بإعادة سحب **الأيام** لا العيّنات."""
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(days, return_inverse=True)
    groups = [np.where(inv == k)[0] for k in range(len(uniq))]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[idx])) < 2:
            continue
        diffs.append(roc_auc_score(y[idx], s_a[idx]) - roc_auc_score(y[idx], s_b[idx]))
    point = roc_auc_score(y, s_a) - roc_auc_score(y, s_b)
    lo, hi = (np.percentile(diffs, [2.5, 97.5]) if diffs else (np.nan, np.nan))
    return point, lo, hi


def _logit(p):
    p = np.clip(np.asarray(p, dtype="float64"), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def candle_baseline_report(model, train_split, val_split, test_split, model_tf=None,
                           targets=("high", "low"), max_train=300_000, n_boot=300, seed=0,
                           batch_size=1024, verbose=True):
    """يقارن رأسَي التصنيف high/low في النموذج بخطوط أساس من شكل الشمعة. يُرجع جدولاً وحكماً لكل هدف."""
    model_tf = model_tf or MODEL_TF
    tr, tr_split = _candle_frame(train_split, model_tf)
    va, va_split = _candle_frame(val_split, model_tf)
    te, te_split = _candle_frame(test_split, model_tf)

    rng = np.random.default_rng(seed)
    tr_idx = np.sort(rng.choice(len(tr), size=min(max_train, len(tr)), replace=False))   # سقف للسرعة/الذاكرة
    X_last = {"train": np.asarray(tr_split[f"X_{model_tf}"][tr_idx, -1, :], dtype="float32"),
              "val": np.asarray(va_split[f"X_{model_tf}"][:, -1, :], dtype="float32"),
              "test": np.asarray(te_split[f"X_{model_tf}"][:, -1, :], dtype="float32")}
    out_val = model.predict(va_split[f"X_{model_tf}"], batch_size=batch_size, verbose=0)
    out_test = model.predict(te_split[f"X_{model_tf}"], batch_size=batch_size, verbose=0)
    frames = {"val": va, "test": te}

    rows, verdicts = [], {}
    for t in targets:
        y_tr = tr[f"y_{t}"].values[tr_idx]
        scores = {"val": {}, "test": {}}

        cpos_lr = LogisticRegression().fit(tr[["cpos"]].values[tr_idx], y_tr)
        candle = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.1,
                                                random_state=seed).fit(tr[_CANDLE_COLS].values[tr_idx], y_tr)
        last = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.1,
                                              random_state=seed).fit(X_last["train"], y_tr)
        for part, df in frames.items():
            scores[part]["rule_cpos"] = cpos_lr.predict_proba(df[["cpos"]].values)[:, 1]
            scores[part]["candle_gbm"] = candle.predict_proba(df[_CANDLE_COLS].values)[:, 1]
            scores[part]["laststep_gbm"] = last.predict_proba(X_last[part])[:, 1]
            out = out_val if part == "val" else out_test
            scores[part]["model_class"] = np.asarray(out[f"y_{t}_class_logits"]).ravel()
            mu = np.asarray(out[f"y_{t}"]).ravel()
            scores[part]["model_reg_mu"] = 1.0 / (1.0 + np.exp(-mu / (np.std(mu) + 1e-12)))  # sign(mu) عند 0.5

        for part, df in frames.items():
            y = df[f"y_{t}"].values
            base = max(y.mean(), 1 - y.mean())
            for name, s in scores[part].items():
                pred = (s >= 0.5).astype(int)
                rows.append({"target": t, "split": part, "score": name, "n": len(y),
                             "auc": roc_auc_score(y, s), "accuracy": float((pred == y).mean()),
                             "balanced_acc": balanced_accuracy_score(y, pred), "majority_baseline": base})

        # ── الحكم على test ──
        y_te, days = te[f"y_{t}"].values, te["timestamp"].values
        base_names = ["rule_cpos", "candle_gbm", "laststep_gbm"]
        best = max(base_names, key=lambda k: roc_auc_score(va[f"y_{t}"].values, scores["val"][k]))  # يُختار على val
        d, lo, hi = _day_bootstrap_auc_diff(y_te, scores["test"]["model_class"], scores["test"][best], days,
                                            n_boot=n_boot, seed=seed)
        # قيمة مضافة: هل تحسّن درجة النموذج أفضل خط أساس حين تُدمَجان؟ (الدمج يُتعلَّم على val فقط)
        y_va = va[f"y_{t}"].values
        Z = lambda part, with_model: np.column_stack(
            [_logit(scores[part][best])] + ([_logit(scores[part]["model_class"])] if with_model else []))
        only_base = LogisticRegression().fit(Z("val", False), y_va)
        combined = LogisticRegression().fit(Z("val", True), y_va)
        s_base = only_base.predict_proba(Z("test", False))[:, 1]
        s_comb = combined.predict_proba(Z("test", True))[:, 1]
        di, loi, hii = _day_bootstrap_auc_diff(y_te, s_comb, s_base, days, n_boot=n_boot, seed=seed + 1)

        if loi > 0:
            verdict = f"✅ النموذج يضيف معلومة فوق {best} (ΔAUC عند الدمج {di:+.4f}، فاصل [{loi:+.4f}, {hii:+.4f}])"
        elif d < 0 or hi < 0:
            verdict = f"❌ {best} يعادل النموذج أو يتفوّق عليه — النموذج لا يضيف شيئاً فوق شكل الشمعة"
        else:
            verdict = f"➖ لا فرق دالّ عن {best} — لا دليل على قيمة مضافة"
        verdicts[t] = {"best_baseline": best, "auc_diff_vs_best": (d, lo, hi),
                       "auc_gain_when_combined": (di, loi, hii), "verdict": verdict,
                       "n_test_days": int(len(np.unique(days)))}

    table = pd.DataFrame(rows)
    if verbose:
        print("\n🕯️ رأسا high/low مقابل خطوط أساس من شكل الشمعة (كل خطوط الأساس دُرِّبت على train فقط)")
        print(f"   فحص تعريف الهدف من الأسعار الخام — val: {va.attrs['label_check']} | test: {te.attrs['label_check']}")
        with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
            print(table.pivot_table(index=["target", "score"], columns="split",
                                    values=["auc", "accuracy"]).round(4).to_string())
            print("\n   majority_baseline:", table.groupby(["target", "split"])["majority_baseline"].first().round(4).to_dict())
        for t, v in verdicts.items():
            d, lo, hi = v["auc_diff_vs_best"]
            print(f"\n   [{t}] أفضل خط أساس (مختار على val): {v['best_baseline']} | "
                  f"AUC النموذج − الأساس على test: {d:+.4f} [{lo:+.4f}, {hi:+.4f}] | أيام test: {v['n_test_days']}")
            print(f"   [{t}] {v['verdict']}")
    return {"table": table, "verdicts": verdicts}


# الاستخدام (بعد تحميل أفضل أوزان):
#   cb = candle_baseline_report(model, train, val, test)

## ٧-د) تحقق متكامل — استدعاء واحد، جدول حكم واحد

```python
ver = run_full_verification(model, train, val, test, out_dir="analysis_outputs")
```

يشغّل بالترتيب ويجمع النتائج في جدول واحد (✅ / ❌ / ➖ لا فرق دالّ / ⚠️ / ℹ️ معلومة)، ويحفظه CSV وJSON:

| القسم | ما يُفحص |
|---|---|
| أ) البيانات | X بلا NaN/Inf؛ أهداف التصنيف والانحدار تطابق الأسعار الخام في `last_candles`؛ train قبل val قبل test بفجوة ≥ نافذة+أفق؛ نسبة الصعود في train |
| ب) خطوط الأساس | دقة اتجاه كل هدف على test مقابل «الفئة الأكبر في train»، وMAE الانحدار مقابل «التنبؤ بصفر» — بفاصل bootstrap يُعيد سحب **الأيام** |
| ج) الثقة والتداول | اتساق الثقة (> +0.5 على val وtest معاً)، شريحة ≥ 65%، صفقات 2:1 — القسم ٧-ب |
| د) شكل الشمعة | هل يضيف رأسا high/low شيئاً فوق قاعدة الشمعة — القسم ٧-ج |

كل عتبة تُختار على train/val فقط، والحكم من test. عدد «أيام test المستقلة» في الجدول يحدّد قوة كل
الأحكام: أقل من ~250 يوماً يعني أن غياب الدلالة لا يثبت غياب الإشارة.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🧾 تحقق متكامل: استدعاء واحد يجيب على كل الأسئلة، بجدول حكم واحد
# ═══════════════════════════════════════════════════════════════════════════
#   run_full_verification(model, train, val, test)
#
# أ) سلامة البيانات (بلا نموذج): قيم منتهية، الأهداف تطابق الأسعار الخام، ترتيب زمني train<val<test
#    بفجوة عزل كافية، وتوازن الفئات.
# ب) خطوط أساس بسيطة على test: اتجاه كل هدف مقابل «الفئة الأكبر في train»، وعائد الانحدار مقابل
#    «التنبؤ بصفر» — بفاصل ثقة bootstrap يُعيد سحب الأيام (عيّنات اليوم الواحد مترابطة).
# ج) الثقة والتداول: selective_direction_report + rr_trading_report (القسم ٧-ب).
# د) شكل الشمعة: candle_baseline_report (القسم ٧-ج).
# كل العتبات تُختار على val/train فقط، والحكم من test. يُحفظ الجدول (CSV/JSON) إن أُعطي out_dir.
import json
import os
import numpy as np
import pandas as pd


def _day_bootstrap_mean(values, days, n_boot=500, seed=0):
    """متوسط قيمة لكل عيّنة، بفاصل 95% بإعادة سحب الأيام (كل يوم بكل عيّناته)."""
    values = np.asarray(values, dtype="float64")
    uniq, inv = np.unique(days, return_inverse=True)
    sums = np.bincount(inv, weights=values)
    counts = np.bincount(inv).astype("float64")
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    boot = sums[draws].sum(1) / counts[draws].sum(1)
    return float(values.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def _all_finite(X, chunk=20000):
    return all(np.isfinite(X[i:i + chunk]).all() for i in range(0, len(X), chunk))


def _split_ts(split_or_dict):
    if "y" in split_or_dict:
        return np.asarray(split_or_dict["last_candles"])[:, _LC["timestamp"]]
    return np.concatenate([np.asarray(s["last_candles"])[:, _LC["timestamp"]] for s in split_or_dict.values()])


def _label_checks(split_or_dict, model_tf):
    split, _ = _concat_splits(split_or_dict, model_tf)
    lc = np.asarray(split["last_candles"], dtype="float64")
    last = {"high": lc[:, _LC["last_high"]], "low": lc[:, _LC["last_low"]], "close": lc[:, _LC["last_close"]]}
    fut = {"high": lc[:, _LC["future_high_max"]], "low": lc[:, _LC["future_low_min"]],
           "close": lc[:, _LC["future_close"]]}
    out = {}
    for t in PRICE_TARGETS:
        if f"y_{t}_class" in split["y"]:
            y = np.asarray(split["y"][f"y_{t}_class"]).ravel() > 0
            out[f"{t}_class"] = float(np.mean(y == (fut[t] > last[t])))
        if f"y_{t}_reg" in split["y"]:
            y = np.asarray(split["y"][f"y_{t}_reg"], dtype="float64").ravel()
            raw = np.clip(fut[t] / last[t] - 1.0, -1.0, 1.0)
            out[f"{t}_reg"] = float(np.mean(np.abs(y - raw) <= 1e-4 + 1e-3 * np.abs(raw)))
    return out


def run_full_verification(model, train_split, val_split, test_split, model_tf=None, target_acc=0.65,
                          rr=2.0, stop_mode="nig", n_boot=500, out_dir=None, run_candle=True,
                          verbose=True):
    """يشغّل كل الفحوص ويُرجع {"summary": جدول الحكم، "details": كل التقارير الفرعية}."""
    model_tf = model_tf or MODEL_TF
    rows, details = [], {}

    def add(section, check, value, verdict, note=""):
        rows.append({"section": section, "check": check, "value": value, "verdict": verdict, "note": note})

    # ── أ) سلامة البيانات ─────────────────────────────────────────────────
    for name, sp in (("train", train_split), ("val", val_split), ("test", test_split)):
        X = sp[f"X_{model_tf}"] if "y" in sp else None
        ok = _all_finite(X) if X is not None else all(_all_finite(s[f"X_{model_tf}"]) for s in sp.values())
        add("أ) البيانات", f"X_{name} بلا NaN/Inf", ok, "✅" if ok else "❌")
    for name, sp in (("val", val_split), ("test", test_split)):
        chk = _label_checks(sp, model_tf)
        worst = min(chk.values()) if chk else 1.0
        add("أ) البيانات", f"الأهداف تطابق الأسعار الخام ({name})", round(worst, 4),
            "✅" if worst >= 0.99 else "❌", json.dumps({k: round(v, 4) for k, v in chk.items()}))
    ts = {n: _split_ts(sp) for n, sp in (("train", train_split), ("val", val_split), ("test", test_split))}
    step = pd.Timedelta(model_tf).value
    window = train_split[f"X_{model_tf}"].shape[1]
    horizon = int(CONFIG.get("forecast_horizon", 1))
    for a, b in (("train", "val"), ("val", "test")):
        gap = (ts[b].min() - ts[a].max()) / step
        add("أ) البيانات", f"{a} قبل {b} زمنياً (فجوة بالشموع)", round(float(gap), 1),
            "✅" if gap >= window + horizon else ("⚠️" if gap > 0 else "❌"),
            f"المطلوب ≥ نافذة+أفق = {window + horizon}")
    for t in PRICE_TARGETS:
        k = f"y_{t}_class"
        if k in train_split["y"]:
            p = float(np.mean(np.asarray(train_split["y"][k]) > 0))
            add("أ) البيانات", f"نسبة الصعود في train ({t})", round(p, 4), "ℹ️", "خط أساس الفئة الأكبر = max(p, 1−p)")

    # ── ب) خطوط الأساس على test ────────────────────────────────────────────
    val_df = collect_signals(model, val_split, model_tf)
    test_df = collect_signals(model, test_split, model_tf)
    te_split, _ = _concat_splits(test_split, model_tf)
    days = test_df["timestamp"].values
    add("ب) خطوط الأساس", "أيام test المستقلة", int(len(np.unique(days))), "ℹ️",
        "كل الأحكام أدناه على هذه الأيام — أقل من ~250 يوماً يعني أحكاماً ضعيفة إحصائياً")
    fut_up = {"close": test_df["fut_close"] > test_df["entry"],
              "high": test_df["fut_high"] > test_df["last_high"],
              "low": test_df["fut_low"] > test_df["last_low"]}
    for t in PRICE_TARGETS:
        if f"p_up_{t}" not in test_df or f"y_{t}_class" not in train_split["y"]:
            continue
        y = fut_up[t].values.astype(int)
        maj = int(np.mean(np.asarray(train_split["y"][f"y_{t}_class"]) > 0) >= 0.5)   # فئة train الأكبر
        pred = (test_df[f"p_up_{t}"].values >= 0.5).astype(int)
        m, lo, hi = _day_bootstrap_mean((pred == y).astype(float) - (maj == y).astype(float), days, n_boot)
        add("ب) خطوط الأساس", f"دقة اتجاه {t} − الفئة الأكبر (test)", round(m, 4),
            "✅" if lo > 0 else ("❌" if hi < 0 else "➖"),
            f"دقة النموذج {np.mean(pred == y):.4f} | الفئة الأكبر {np.mean(maj == y):.4f} | فاصل [{lo:+.4f}, {hi:+.4f}]")
    for t in PRICE_TARGETS:
        if f"y_{t}_reg" not in te_split["y"] or f"mu_{t}" not in test_df:
            continue
        y = np.asarray(te_split["y"][f"y_{t}_reg"], dtype="float64").ravel()
        gain = np.abs(y) - np.abs(y - test_df[f"mu_{t}"].values)          # > 0 = النموذج أقرب من الصفر
        m, lo, hi = _day_bootstrap_mean(gain, days, n_boot)
        add("ب) خطوط الأساس", f"MAE الصفر − MAE النموذج ({t}_reg، test)", round(m, 5),
            "✅" if lo > 0 else ("❌" if hi < 0 else "➖"),
            f"MAE النموذج {np.mean(np.abs(y - test_df[f'mu_{t}'].values)):.5f} | الصفر {np.mean(np.abs(y)):.5f}"
            f" | فاصل [{lo:+.5f}, {hi:+.5f}]")

    # ── ج) الثقة والتداول ─────────────────────────────────────────────────
    sel = selective_direction_report(val_df, test_df, target_acc=target_acc, verbose=verbose)
    trd = rr_trading_report(val_df, test_df, rr=rr, stop_mode=stop_mode, verbose=verbose)
    details.update({"selective": sel, "trades": trd, "val_df": val_df, "test_df": test_df})
    for _, r in sel.iterrows():
        cons = (r["monotonic_val"] > 0.5) and (r["monotonic_test"] > 0.5)
        add("ج) الثقة", f"اتساق الثقة ({r['score']})", f"val {r['monotonic_val']:+.2f} / test {r['monotonic_test']:+.2f}",
            "✅" if cons else "❌", "المطلوب > +0.5 على الفترتين معاً")
        add("ج) الثقة", f"شريحة ≥ {target_acc:.0%} ({r['score']})", r.get("test_acc", np.nan),
            "✅" if str(r["verdict"]).startswith("✅") else "❌", str(r["verdict"]))
    for _, r in trd.iterrows():
        add("ج) التداول", f"صفقات {rr:g}:1 ({r['score']})", r.get("test_exp_R", np.nan),
            "✅" if str(r["verdict"]).startswith("✅") else "❌",
            f"val_exp_R={r.get('val_exp_R', np.nan):.3f}, t_days={r.get('test_t_days', np.nan):.2f}")

    # ── د) شكل الشمعة ─────────────────────────────────────────────────────
    if run_candle:
        cb = candle_baseline_report(model, train_split, val_split, test_split, model_tf, verbose=verbose)
        details["candle"] = cb
        for t, v in cb["verdicts"].items():
            add("د) شكل الشمعة", f"{t}_class فوق {v['best_baseline']}",
                round(v["auc_gain_when_combined"][0], 4), v["verdict"][:1], v["verdict"])

    summary = pd.DataFrame(rows)
    if verbose:
        print("\n" + "═" * 100 + "\n🧾 ملخص التحقق المتكامل\n" + "═" * 100)
        with pd.option_context("display.max_colwidth", 90, "display.width", 250):
            print(summary.to_string(index=False))
        n_ok = int((summary["verdict"] == "✅").sum())
        n_bad = int((summary["verdict"] == "❌").sum())
        print(f"\n   ✅ {n_ok} | ❌ {n_bad} | ➖/⚠️/ℹ️ {len(summary) - n_ok - n_bad}")
        trade_ok = summary[summary["section"].isin(["ج) الثقة", "ج) التداول"])]["verdict"].eq("✅").any()
        print("   الخلاصة:", "يوجد على الأقل مصدر ثقة/صفقات اجتاز test — افحصه على نوافذ زمنية أخرى قبل أي استخدام"
              if trade_ok else "لا شيء قابل للتداول اجتاز test بهذا النموذج وهذه البيانات")
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        summary.to_csv(os.path.join(out_dir, "verification_summary.csv"), index=False, encoding="utf-8-sig")
        with open(os.path.join(out_dir, "verification_summary.json"), "w", encoding="utf-8") as f:
            json.dump(summary.astype(str).to_dict("records"), f, ensure_ascii=False, indent=1)
        if verbose:
            print(f"   💾 حُفظ في {out_dir}/verification_summary.csv و.json")
    return {"summary": summary, "details": details}


# الاستخدام (بعد تحميل أفضل أوزان):
#   ver = run_full_verification(model, train, val, test, out_dir="analysis_outputs")

## ٨) اختبار ذاتي للتوصيل بين الدفاتر (بيانات تركيبية — بلا Drive ولا تدريب حقيقي)

يبني بيانات بنفس شكل مخرَج خط الأنابيب تماماً (بما فيها ترميز التصنيف
الحقيقي `+1.0`/`-1.0`، لا `{0,1}` — القسم ٠.٤)، يُدرِّب نموذجاً حقيقياً
حقبتين، ثم يُغذّي `chicks` عبر نفس `build_chicks_test_dict` أعلاه — يُثبت
أن كل نقاط الربط الأربع (القسم ٠) تعمل معاً فعلياً، لا افتراضاً.
شغّله بعد أي تعديل على قسم الربط في هذا الدفتر، أو على أي من الدفاتر الأربعة.


In [ ]:
def run_wiring_selftest(verbose=True):
    # يثبّت التحويل نفسه أولاً — تأكيد حتمي بمعزل عن عشوائية التدريب
    assert _to_unit_label(np.array([-1.0, 1.0])).tolist() == [0.0, 1.0], "_to_unit_label خاطئة"

    rng = np.random.default_rng(0)
    seq_len, n_features = 16, 10
    tf_name = "1D"

    def make_split(n):
        X = rng.normal(size=(n, seq_len, n_features)).astype("float32")
        last_close = 100 + rng.normal(size=(n,)) * 5
        last_high = last_close + np.abs(rng.normal(size=(n,)))
        last_low = last_close - np.abs(rng.normal(size=(n,)))
        ret = {t: rng.normal(scale=0.01, size=(n,)).astype("float32") for t in ("high", "low", "close")}
        # ترميز خط الأنابيب الحقيقي هو +1.0/-1.0 (اتجاه)، لا {0,1} — القسم ٠.٤
        cls = {t: (rng.integers(0, 2, size=(n,)).astype("float32") * 2 - 1) for t in ("high", "low", "close")}
        last_candles = np.stack([
            last_high, last_low, last_close, np.arange(n).astype("float64"),
            last_close * (1 + ret["close"]), last_low * (1 + ret["low"]), last_high * (1 + ret["high"]),
        ], axis=1)
        y = {f"y_{t}_reg": v for t, v in ret.items()}
        y.update({f"y_{t}_class": v for t, v in cls.items()})
        return {
            f"X_{tf_name}": X,
            "y": y,
            "base_params": np.zeros((n, 2), dtype="float32"),
            "last_candles": last_candles,
        }

    fake_train, fake_val = make_split(256), make_split(64)
    fake_test = {"A": make_split(50), "B": make_split(40)}

    fake_model_builder = lambda: build_model_fn(seq_len, n_features)
    fake_config = build_config({
        "run": {"run_dir": "/tmp/_wiring_selftest_run", "epochs": 1, "batch_size": 32,
                "verbose": 0, "train_mode": "new"},
        "targets": build_target_configs(("high", "low", "close")),
    })
    fake_train_ds = tf.data.Dataset.from_tensor_slices(
        (fake_train[f"X_{tf_name}"], _y_for_targets(fake_train, fake_config))
    ).batch(32, drop_remainder=True)
    fake_val_ds = tf.data.Dataset.from_tensor_slices(
        (fake_val[f"X_{tf_name}"], _y_for_targets(fake_val, fake_config))
    ).batch(32, drop_remainder=True)
    sample = next(iter(fake_train_ds))

    fake_trainer, fake_callbacks, fake_ie = build_training_system(fake_model_builder, fake_config, sample)
    fake_history = fake_trainer.fit(
        fake_train_ds, validation_data=fake_val_ds, initial_epoch=fake_ie,
        epochs=fake_config["run"]["epochs"], callbacks=fake_callbacks, verbose=0)
    fake_model = fake_trainer.model

    # التحقّق أن أهداف التصنيف فعلياً دخلت التدريب (مقياس accuracy مُسجَّل لكل منها)
    for t in ("high", "low", "close"):
        assert any(k.startswith(f"{t}_class") and "accuracy" in k for k in fake_history.history), (
            f"لا مقياس accuracy لهدف {t}_class — رأس التصنيف لم يُدرَّب فعلياً")

    fake_test_dict = {}
    for asset, split in fake_test.items():
        last_close = split["last_candles"][:, LAST_CLOSE_COL]
        fake_test_dict[asset] = {
            f"X_{tf_name}": split[f"X_{tf_name}"],
            "base_params": np.stack([last_close, last_close], axis=1).astype("float32"),
            "last_candles": split["last_candles"],
            "y": {t: split["y"][f"y_{t}_reg"] for t in ("high", "low", "close")},
        }

    results = run_full_analysis(
        model=fake_model, test_dict=fake_test_dict, timeframes=[tf_name],
        target_specs=EVAL_TARGET_SPECS, make_plots=False, verbose=False,
        out_dir="/tmp/_wiring_selftest_analysis",
    )
    assert "per_asset_results" in results and len(results["per_asset_results"]) == 2

    # التقييم الانتقائي (القسم ٧-ب) على مخرجات النموذج الفعلية وبنية val (قسم واحد) وtest (قاموس أصول)
    sel = selective_evaluation(fake_model, fake_val, fake_test, tf_name, min_n=10, verbose=False)
    assert len(sel["val_df"]) == 64 and len(sel["test_df"]) == 90
    assert {"class_margin", "nig_edge", "conf_head", "agree_margin"} <= set(sel["direction"]["score"])

    # خطوط أساس شكل الشمعة (القسم ٧-ج) على نفس البيانات الوهمية
    cb = candle_baseline_report(fake_model, fake_train, fake_val, fake_test, tf_name, n_boot=20, verbose=False)
    assert set(cb["verdicts"]) == {"high", "low"} and len(cb["table"]) == 2 * 2 * 5

    # التحقق المتكامل (القسم ٧-د) — يعمل من طرفه لطرفه على مخرجات النموذج الفعلية
    ver = run_full_verification(fake_model, fake_train, fake_val, fake_test, tf_name, n_boot=50, verbose=False)
    assert {"أ) البيانات", "ب) خطوط الأساس", "ج) الثقة", "ج) التداول", "د) شكل الشمعة"} <= set(ver["summary"]["section"])

    # الدفعة المدمجة (predict_pooled_batch_by_asset): يجب أن تُعطي نتائج مطابقة عملياً
    # لاستدعاء منفصل لكل أصل (predict_latest_all_assets) رغم أنها تستدعي predict_batch_v4
    # مرّة واحدة فقط لكل الأصول معاً بدل استدعاء منفصل لكل أصل — التحقّق هنا رقمي
    # (تطابق القيم)؛ استدعاء النموذج مرّة واحدة فقط خاصية بنيوية واضحة من كود الدالة نفسها.
    # تسامح (rtol/atol) بدل التطابق الحتمي: تركيب الدفعة يُغيّر ترتيب عمليات الجمع
    # العائم (matmul/reduction) فيُنتج فروقاً دقيقة (~1e-5 على قيم بمئات) لا علاقة
    # لها بصحّة الحساب — النموذج لا يحوي BatchNorm فهو مستقلّ عن تركيب الدفعة رياضياً.
    pooled, y_true_pooled = pool_test_dict(fake_test_dict, tf_name)
    assert [b["name"] for b in pooled["asset_bounds"]] == list(fake_test_dict.keys())
    pooled_table = predict_pooled_batch_by_asset(
        fake_model, pooled, tf_name, target_specs=EVAL_TARGET_SPECS,
        n_display=1000, y_true_pooled=y_true_pooled, verbose=False)
    separate_table = predict_latest_all_assets(
        fake_model, fake_test_dict, timeframes=[tf_name], target_specs=EVAL_TARGET_SPECS,
        n_display=1000, verbose=False)
    for asset in fake_test_dict:
        for t in ("high", "low", "close"):
            a = pooled_table.query("asset == @asset and target == @t")["pred"].to_numpy()
            b = separate_table.query("asset == @asset and target == @t")["pred"].to_numpy()
            assert a.shape == b.shape and np.allclose(a, b, rtol=1e-3, atol=1e-2, equal_nan=True), (
                f"الدفعة المدمجة تختلف عن الحلقة اليدوية لـ{asset}/{t}")
    if verbose:
        print("  ✅ predict_pooled_batch_by_asset: نتائج مطابقة (بحدود دقّة float32) لاستدعاء منفصل لكل أصل، "
              "باستدعاء نموذج واحد فقط للأصلين معاً بدل استدعاء لكل أصل")

    # مسار التصنيف المنفصل (classification_accuracy_report) بنفس المنطق، على البيانات الوهمية —
    # بترميز +1.0/-1.0 المُحوَّل عبر _to_unit_label، تماماً كما في classification_accuracy_report الفعلية
    from sklearn.metrics import accuracy_score
    for asset, split in fake_test_dict.items():
        out = fake_model(split[f"X_{tf_name}"], training=False)
        for t in ("high", "low", "close"):
            y_true = _to_unit_label(np.asarray(fake_test[asset]["y"][f"y_{t}_class"]))
            assert set(np.unique(y_true).tolist()) <= {0.0, 1.0}, "التحويل لم ينتج {0,1}"
            y_prob = out[f"y_{t}_class_logits"].numpy().ravel()
            accuracy_score(y_true, (y_prob >= 0.5).astype("float32"))  # لا يرفع استثناءً يكفي هنا

    if verbose:
        print("✅ نجح اختبار التوصيل: بيانات ← نموذج (انحدار+تصنيف، بترميز +1/-1 الحقيقي) ← "
              f"تدريب ← chicks + تقرير تصنيف مستقلّ، بلا أي استثناء، عبر {len(results)} تقريراً من chicks")
    return True


def _y_for_targets(split, config):
    y = {}
    for cfg in config["targets"].values():
        v = split["y"][cfg["true_key"]]
        y[cfg["true_key"]] = _to_unit_label(v) if cfg["task_type"] == "classification" else v
    return y


run_wiring_selftest()